In [ ]:
# Determinism vs Random Seeds

## Goal

Understand why LLM outputs can differ across runs.

We compare:

1) **Greedy decoding** (do_sample=False)
   -> deterministic outputs

2) **Sampling** (do_sample=True)
   -> outputs depend on randomness and seeds

We keep everything constant:
- same model
- same prompt
- same decoding parameters

Only the seed changes.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()

In [ ]:
prompt = """
Explain photosynthesis.

Return exactly 5 bullet points.
Each bullet must:
- Start with "- "
- Contain at most 12 words
- Use simple vocabulary
""".strip()

In [ ]:
def generate(prompt, do_sample, seed=0, temperature=1.0, top_p=0.9, max_new_tokens=120):
    set_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
print("=== Greedy decoding (do_sample=False) ===\n")
print(generate(prompt, do_sample=False, seed=0))
print("\n--- Same run again ---\n")
print(generate(prompt, do_sample=False, seed=999))

In [ ]:
print("\n=== Sampling decoding (do_sample=True) ===\n")

for i in [0, 1, 2]:
    print(f"\n--- Seed {i} ---")
    print(generate(prompt, do_sample=True, seed=i, temperature=1.0, top_p=0.9))

In [ ]:
## Observations

- Greedy decoding is deterministic: seeds do not matter.
- Sampling introduces randomness: changing the seed changes the output.
- With the same seed and settings, outputs are reproducible.

## Key Insight

Output variability is not a "model mood".
It is a property of stochastic decoding.

If you want reproducible results in experiments:
- fix the seed
- log decoding settings
- keep the environment stable

In production:
- deterministic decoding increases reliability
- sampling increases diversity but reduces reproducibility